# CFD discovery with SCFDM

Conditional functional dependencies carry *pattern tableaus*:
`[city] => state, (New York || NY)` holds only for matching tuples; `_` is a wildcard.

SCFDM = RepSampler (MinHash-LSH sampling with statistical bounds) → correlation
extraction (Transformer AttrFinder, or lightweight fallback) → vertical
partitioning → native CFD kernel with 10 search strategies.

In [1]:
import pandas as pd
import nesydep as nd

df = pd.read_csv('../tests/data/census42_sample.csv')
print(df.shape)

(500, 42)


## Discover CFDs (no GPU needed)

`correlation='lightweight'` uses Pearson association; with the `[transformer]`
extra installed you can use `correlation='transformer'` (AttrFinder).

In [2]:
miner = nd.get_algorithm('scfdm')(correlation='lightweight', support=5, confidence=0.95, max_lhs=2)
result = miner.discover(df)
print(f'{len(result)} CFDs from {result.stats["n_subtables"]} sub-tables')
result.cfds[:5]

[1/4] sampling (500 rows) ...


1696 CFDs from 19 sub-tables


[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (19 sub-tables) ...


[CFD(lhs=('age',), rhs='detailed household and family stat', lhs_pattern=('17',), rhs_pattern='Child <18 never marr not in subfamily'),
 CFD(lhs=('age',), rhs='detailed household and family stat', lhs_pattern=('5',), rhs_pattern='Child <18 never marr not in subfamily'),
 CFD(lhs=('age',), rhs='detailed household and family stat', lhs_pattern=('7',), rhs_pattern='Child <18 never marr not in subfamily'),
 CFD(lhs=('age',), rhs='detailed household and family stat', lhs_pattern=('11',), rhs_pattern='Child <18 never marr not in subfamily'),
 CFD(lhs=('age',), rhs='detailed household and family stat', lhs_pattern=('4',), rhs_pattern='Child <18 never marr not in subfamily')]

## Constant vs variable CFDs

In [3]:
constant = [c for c in result.cfds if c.is_constant]
variable = [c for c in result.cfds if not c.is_constant]
print(f'{len(constant)} constant, {len(variable)} variable CFDs')
print('constant example:', constant[0] if constant else None)
print('variable example:', variable[0] if variable else None)

1618 constant, 78 variable CFDs
constant example: [age] => detailed household and family stat, (17 || Child <18 never marr not in subfamily)
variable example: [wage per hour] => member of a labor union, (_ || _)


## Verify against the full dataset

Support/confidence are re-checked with an independent pandas implementation
(CTane tuple-removal semantics for variable CFDs).

In [4]:
from nesydep.evaluation.verify import verify
from nesydep.core.dataset import as_categorical_view, as_dataset

frame = as_categorical_view(as_dataset(df))
checks = verify(frame, result.cfds, min_support=5, min_confidence=0.95)
print(f'{sum(v.holds for v in checks)}/{len(checks)} CFDs hold on the sample')

1696/1696 CFDs hold on the sample


## Compare search strategies

The kernel offers 10 strategies in two families: FD-First (finer, more constant
CFDs) and CTane-style Itemset-First/Integrated (coarser cover). `algo='ctane'`
runs the classic CTane baseline directly.

In [5]:
ctane = nd.discover(df, algo='ctane', support=5, confidence=0.95, max_lhs=2)
print(f'CTane baseline: {len(ctane)} CFDs')
result.to_legacy_txt('cfds.txt')

CTane baseline: 285 CFDs


[1/4] sampling (500 rows) ...
[2/4] correlation extraction ...
[3/4] partitioning ...
[4/4] mining (1 sub-tables) ...
